# Discrete Flow Matching (DFM) for Cell Type Proportion Dynamics

This notebook trains (or loads) a **global shared** DFM denoiser to predict one-step cell type distributions between adjacent timepoints in an scRNA-seq dataset.

Key settings (baseline for 1-day delivery):
- U-coupling
- Replacement path with $\kappa(s)=s$
- Global shared denoiser conditioned on $(s, t_{start}, \Delta t)$


In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

import sys
sys.path.append('../')  

from utils import load_config, set_seed, get_device, read_rdata_dataframe, build_label_mapping, build_pools_by_time, compute_time_encoding
from models import DFMModelConfig, DFMTimeDenoiser
from train_dfm import evaluate_one_step_all_intervals
from utils import empirical_distribution, propagate_distribution_deterministic, js_divergence, l1_distance, kl_divergence, ensure_dir
from eval import compute_or_load_c_full, HeldOutEvalConfig, evaluate_held_out


In [3]:
CONFIG_PATH = Path("../config/dfm_emcc_config.json")
cfg = load_config(CONFIG_PATH)

import json as _json
HELD_OUT_K = cfg.get("held_out", False)

cfg = _json.loads(_json.dumps(cfg)) 
cfg["held_out"] = HELD_OUT_K

experiment_name = str(cfg.get("experiment_name", "default") or "default")
base_out_dir = Path(cfg["paths"]["out_dir"])
exp_out_dir = ensure_dir(base_out_dir / experiment_name)

# Write an "effective" config snapshot for reproducibility (and for subprocess training)
CONFIG_PATH_EFFECTIVE = exp_out_dir / "config_effective.json"
CONFIG_PATH_EFFECTIVE.write_text(_json.dumps(cfg, indent=2), encoding="utf-8")
print("Effective config:", CONFIG_PATH_EFFECTIVE)
print("Held-out:", HELD_OUT_K)

def resolve_exp_path(path_value, default_name):
    p = Path(path_value) if path_value is not None else Path(default_name)
    if p.is_absolute():
        return p
    try:
        relative = p.relative_to(base_out_dir)
    except ValueError:
        relative = p
    return exp_out_dir / relative

checkpoint_path = resolve_exp_path(cfg["paths"].get("checkpoint_path", "dfm_last.pth"), "dfm_last.pth")
best_checkpoint_path = resolve_exp_path(cfg["paths"].get("best_checkpoint_path", "dfm_best.pth"), "dfm_best.pth")
loss_history_path = resolve_exp_path(cfg["train"].get("loss_history_path", "loss_history.json"), "loss_history.json")
held_out_eval_path = resolve_exp_path(cfg.get("eval", {}).get("held_out_eval_path", "held_out_eval.json"), "held_out_eval.json")

print("Experiment:", experiment_name)
print("Checkpoint (last):", checkpoint_path)
print("Checkpoint (best):", best_checkpoint_path)
print("Loss history:", loss_history_path)
print("Held-out eval:", held_out_eval_path)
print("RData path:", cfg["paths"]["rdata_path"])

set_seed(int(cfg.get("seed", 0)))
device = get_device(cfg.get("device", "auto"))
print("Device:", device)



Effective config: outputs_dfm\emcc\config_effective.json
Held-out: False
Experiment: emcc
Checkpoint (last): outputs_dfm\emcc\dfm_last.pth
Checkpoint (best): outputs_dfm\emcc\dfm_best.pth
Loss history: outputs_dfm\emcc\loss_history.json
Held-out eval: outputs_dfm\emcc\held_out_eval.json
RData path: ../data/EMCC.RData
Device: cuda


In [4]:

# Load dataset from .RData
df = read_rdata_dataframe(cfg["paths"]["rdata_path"], key=cfg["data"].get("rdata_key", "Data"))
required = {"celltype", "typeName", "timepoint"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"DataFrame missing columns: {missing}. Present: {list(df.columns)}")

raw_to_idx, idx_to_raw = build_label_mapping(df, label_col=cfg["data"].get("label_col", "celltype"))
timepoints, pools = build_pools_by_time(
    df,
    time_col=cfg["data"].get("time_col", "timepoint"),
    label_col=cfg["data"].get("label_col", "celltype"),
    raw_to_idx=raw_to_idx,
)
time_enc = compute_time_encoding(timepoints)
num_states = len(raw_to_idx)

print("rows:", len(df))
print("timepoints:", timepoints)
print("num_states (K):", num_states)


rows: 6316
timepoints: [11. 13. 15. 17.]
num_states (K): 7


In [5]:
mcfg = cfg["model"]
model_cfg = DFMModelConfig(
    num_states=num_states,
    emb_dim=int(mcfg.get("emb_dim", 64)),
    s_condition=bool(mcfg.get("s_condition", True)),
    t_condition=bool(mcfg.get("t_condition", True)),
    time_mlp_hidden=int(mcfg.get("time_mlp_hidden", 64)),
    trunk_hidden=int(mcfg.get("trunk_hidden", 128)),
    trunk_layers=int(mcfg.get("trunk_layers", 3)),
    dropout=float(mcfg.get("dropout", 0.0)),
)
model = DFMTimeDenoiser(model_cfg).to(device)

if checkpoint_path.exists():
    ckpt = torch.load(str(checkpoint_path), map_location="cpu")
    try:
        model.load_state_dict(ckpt["model_state"])
        print(f"Loaded checkpoint: {checkpoint_path}")
    except Exception as e:
        print(f"Found checkpoint but failed to load (likely architecture change): {type(e).__name__}: {e}")
        print("Please retrain to generate a compatible checkpoint.")
else:
    print("No checkpoint found for this experiment. Train first (next cell).")


Loaded checkpoint: outputs_dfm\emcc\dfm_last.pth


In [6]:
import subprocess, sys

do_train = True  
if do_train:
    cmd = [sys.executable, "-u", "../train_dfm.py", "--config", str(CONFIG_PATH_EFFECTIVE)]
    print("Running:", " ".join(cmd))
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as proc:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
        retcode = proc.wait()
    if retcode != 0:
        raise subprocess.CalledProcessError(retcode, cmd)


Running: c:\envs\torch_env\Scripts\python.exe -u ../train_dfm.py --config outputs_dfm\emcc\config_effective.json
[Resume] Loaded checkpoint: outputs_dfm\emcc\dfm_last.pth (epoch=15, step=3000, train_time_seconds=15.21)
d:\PKU\zpj\ICML2026\to gpt\StateFlow\train_dfm.py:551: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
[Export] Saved transition matrices to outputs_dfm\emcc\transition_matrix using checkpoint outputs_dfm\emcc\dfm_best.pth


In [7]:
if best_checkpoint_path.exists():
    ckpt = torch.load(str(best_checkpoint_path), map_location="cpu")
    try:
        model.load_state_dict(ckpt["model_state"])
        print(f"Loaded best checkpoint: {best_checkpoint_path}")
    except Exception as e:
        print(f"Found best checkpoint but failed to load (likely architecture change): {type(e).__name__}: {e}")
        print("Please retrain to generate a compatible checkpoint.")
model.eval()


Loaded best checkpoint: outputs_dfm\emcc\dfm_best.pth


DFMTimeDenoiser(
  (state_emb): Embedding(7, 64)
  (time_mlp): Sequential(
    (0): Linear(in_features=3, out_features=64, bias=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): SiLU()
  )
  (trunk): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): SiLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): SiLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): SiLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=128, out_features=7, bias=True)
  )
)

## held_out = False

In [16]:

cfg_eval = load_config(CONFIG_PATH)
df_eval = read_rdata_dataframe(cfg_eval["paths"]["rdata_path"], key=cfg_eval.get("data", {}).get("rdata_key", "Data"))
raw_to_idx, idx_to_raw = build_label_mapping(df_eval, label_col=cfg_eval.get("data", {}).get("label_col", "celltype"))
timepoints, pools = build_pools_by_time(df_eval, time_col=cfg_eval.get("data", {}).get("time_col", "timepoint"), label_col=cfg_eval.get("data", {}).get("label_col", "celltype"), raw_to_idx=raw_to_idx)
time_enc = compute_time_encoding(timepoints)
num_states = len(raw_to_idx)
cost_cfg = cfg_eval.get("coupling", {}).get("cost", {})
tsne_cols = cost_cfg.get("cols", ["tSNE_1", "tSNE_2"])
centroid_stat = cost_cfg.get("centroid_stat", "median")
cost_power = cost_cfg.get("power", 2)
cost_normalize = cost_cfg.get("normalize", "median_offdiag")
c_full_cache = Path(cfg_eval.get("eval", {}).get("c_full_cache_path", "C_full.npy"))
C_full = compute_or_load_c_full(df_full=df_eval, raw_to_idx=raw_to_idx, tsne_cols=tsne_cols, centroid_stat=centroid_stat, cost_power=cost_power, cost_normalize=cost_normalize, cache_path=c_full_cache)
best_ckpt = Path(cfg_eval["paths"].get("best_checkpoint_path", "outputs_dfm/dfm_best.pth"))
if not best_ckpt.is_absolute():
    best_ckpt = Path(cfg_eval["paths"].get("out_dir", "outputs_dfm")) / cfg_eval.get("experiment_name", "default") / best_ckpt
ckpt = torch.load(str(best_ckpt), map_location="cpu")
model = DFMTimeDenoiser(DFMModelConfig(num_states=num_states, **{k:v for k,v in cfg_eval.get("model", {}).items() if k in DFMModelConfig.__annotations__}))
model.load_state_dict(ckpt["model_state"]); model.eval()
device = get_device(cfg_eval.get("device", "auto")); model.to(device)
num_steps_eval = int(cfg_eval.get("train", {}).get("num_steps_eval", 20))
smoothing = float(cfg_eval.get("data", {}).get("smoothing", 1e-6))
metrics = evaluate_one_step_all_intervals(
    model=model,
    pools=pools,
    timepoints=timepoints,
    time_enc=time_enc,
    num_states=num_states,
    num_steps_eval=num_steps_eval,
    device=device,
    smoothing=smoothing,
    cost_matrix=C_full,
    sinkhorn_eps=float(cfg_eval.get("eval", {}).get("sinkhorn_eps", 0.1)),
    sinkhorn_iters=int(cfg_eval.get("eval", {}).get("sinkhorn_iters", 300)),
    support_only=bool(cfg_eval.get("eval", {}).get("support_only", False)),
    smoothing_eps=float(cfg_eval.get("eval", {}).get("smoothing_eps", 1e-8)),
)
metrics


{'js_mean': 0.00018821868101843412,
 'kl_mean': 0.0007519973225103832,
 'l1_mean': 0.02941961114201205,
 'js_std': 0.00014705209196721013,
 'kl_std': 0.0005867550007936459,
 'l1_std': 0.012295408395450508,
 'w1_mean': 0.018239049952975914,
 'w1_std': 0.009312768772110598}

## held_out = k

In [ ]:

cfg_eval = load_config(CONFIG_PATH)
df_full = read_rdata_dataframe(cfg_eval["paths"]["rdata_path"], key=cfg_eval.get("data", {}).get("rdata_key", "Data"))
raw_to_idx, idx_to_raw = build_label_mapping(df_full, label_col=cfg_eval.get("data", {}).get("label_col", "celltype"))
timepoints_full, pools_full = build_pools_by_time(df_full, time_col=cfg_eval.get("data", {}).get("time_col", "timepoint"), label_col=cfg_eval.get("data", {}).get("label_col", "celltype"), raw_to_idx=raw_to_idx)
time_enc_train = compute_time_encoding(timepoints_full)
num_states = len(raw_to_idx)
cost_cfg = cfg_eval.get("coupling", {}).get("cost", {})
tsne_cols = cost_cfg.get("cols", ["tSNE_1", "tSNE_2"])
centroid_stat = cost_cfg.get("centroid_stat", "median")
cost_power = cost_cfg.get("power", 2)
cost_normalize = cost_cfg.get("normalize", "median_offdiag")
c_full_cache = Path(cfg_eval.get("eval", {}).get("c_full_cache_path", "C_full.npy"))
C_full = compute_or_load_c_full(df_full=df_full, raw_to_idx=raw_to_idx, tsne_cols=tsne_cols, centroid_stat=centroid_stat, cost_power=cost_power, cost_normalize=cost_normalize, cache_path=c_full_cache)
held_out = cfg_eval.get("held_out", False)
best_ckpt = Path(cfg_eval["paths"].get("best_checkpoint_path", "outputs_dfm/dfm_best.pth"))
if not best_ckpt.is_absolute():
    best_ckpt = Path(cfg_eval["paths"].get("out_dir", "outputs_dfm")) / cfg_eval.get("experiment_name", "default") / best_ckpt
ckpt = torch.load(str(best_ckpt), map_location="cpu")
model = DFMTimeDenoiser(DFMModelConfig(num_states=num_states, **{k:v for k,v in cfg_eval.get("model", {}).items() if k in DFMModelConfig.__annotations__}))
model.load_state_dict(ckpt["model_state"]); model.eval()
device = get_device(cfg_eval.get("device", "auto")); model.to(device)
if held_out not in (False, None, 0, "false", "False"):
    held_out_k = int(held_out)
    metrics_ho = evaluate_held_out(
        model=model,
        held_out_k=held_out_k,
        timepoints_full=timepoints_full,
        pools_full=pools_full,
        timepoints_train=timepoints_full,
        pools_train=pools_full,
        time_enc_train=time_enc_train,
        num_states=num_states,
        device=device,
        C_full=C_full,
        cfg=HeldOutEvalConfig(
            sinkhorn_eps=float(cfg_eval.get("eval", {}).get("sinkhorn_eps", 0.1)),
            sinkhorn_iters=int(cfg_eval.get("eval", {}).get("sinkhorn_iters", 300)),
            support_only=bool(cfg_eval.get("eval", {}).get("support_only", False)),
            smoothing_eps=float(cfg_eval.get("eval", {}).get("smoothing_eps", 1e-8)),
            interp_steps=int(cfg_eval.get("eval", {}).get("interp_steps", 50)),
            extrap_steps=int(cfg_eval.get("eval", {}).get("extrap_steps", int(cfg_eval.get("train", {}).get("num_steps_eval", 25)))),
        ),
        data_smoothing=float(cfg_eval.get("data", {}).get("smoothing", 1e-6)),
    )
    metrics_ho
else:
    print("held_out=False")
